# Optimización de hiperparámetros MDLF con Optuna — v2 LINE · 75mil

**Datos**: almacén `75mil_tarjetas_enero_junio_2025`  
**Split**: temporal — train: enero–mayo 2025 / test: junio 2025 (`split_date = 2025-06-01`)  
**Estrategia**: 150 trials × 7 épocas (proxy), luego reentrenamiento con 30 épocas de la mejor config  
**Objetivo**: maximizar Top-1 accuracy en test

### Diferencias respecto a `mdlf_params_line.ipynb` (50mil · v3)

| Aspecto | 50mil (v3 LINE) | 75mil (v2 LINE) |
|---|---|---|
| Almacén | `50mil_tarjetas_3_al_31_marzo` | `75mil_tarjetas_enero_junio_2025` |
| Período | marzo 2025 (1 mes) | enero–junio 2025 (6 meses) |
| `split_date` | 2025-03-24 (última semana de marzo como test) | 2025-06-01 (junio completo como test) |
| `TTP_PATHS` | períodos 01–03 | períodos 01–06 |
| `holiday_dates` | **no se pasaba** (marzo no tiene festivos nacionales) | **7 festivos** enero–mayo |
| `n_weeks` | suggest_int(1, 4) | suggest_int(1, 8) — más historial disponible |
| Hiperparámetros LINE | variables | **fijos** (mejor config de `optuna_line_cercanias`) |
| Otros hiperparámetros fijos | 7 (los 100% consistentes en top-15 v2) | **16 en total** (LINE + 9 con evidencia cruzada v2+v3) |

### Hiperparámetros fijos

#### LINE — fijos desde `optuna_line_cercanias`
Optimizados de forma independiente maximizando la correlación de Spearman entre distancia coseno en el embedding y tiempo de viaje en el grafo (score 0.7949, Spearman r final 0.9169 con 5 seeds). **LINE se entrena una sola vez** antes del bucle de Optuna.

| Parámetro | Valor |
|---|---|
| `dim` | 64 |
| `asymmetric` | False |
| `num_negative` | 2 |
| `lr` | 0.009611 |
| `num_epochs_fo` | 850 |
| `num_epochs_so` | 200 |
| `init_scale` | 0.038254 |

#### Resto — fijos por evidencia cruzada v2 · v3 (50mil)
Parámetros que aparecen de forma consistente en el top-15 de dos estudios independientes (ver celda siguiente para la justificación completa).

| Grupo | Parámetro | Valor fijo |
|---|---|---|
| Norm | `norm` | `None` |
| LDA | `lda_use_periods` | `False` |
| LDA | `lda_token_granularity` | `'temporal'` |
| LDA | `lda_alpha_none` | `False` (alpha siempre explícita) |
| Ext. | `include_prox` | `True` |
| Train | `lr_scheduler` | `True` |
| CNN | `kernel_size` | `3` |
| CNN | `cnn_channels` | `32` |
| CNN | `d_t` | `8` |

### Parámetros buscados por Optuna (11 dimensiones)

| Grupo | Parámetros | Rango / opciones |
|---|---|---|
| LDA | `n_topics` | [8, 16, 32, 48] |
| LDA | `lda_alpha` | float log [0.001, 1.0] |
| LDA | `lda_beta_none` | [True, False] |
| LDA | `lda_beta` | float log [0.001, 1.0] (si beta_none=False) |
| LDA | `lda_log_counts` | [True, False] |
| LDA | `lda_min_df` | int [1, 5] |
| LDA | `lda_min_station_trips` | [0, 50, 100] |
| CNN | `n_heads` | [2, 4, 8] |
| Train | `optimizer` | ['sgd', 'adamw'] |
| Train | `lr` | float log [1e-3, 1e-1] |
| Train | `weight_decay` | float log [1e-4, 1e-1] |
| Train | `n_weeks` | int [1, 8] |
| Ext. | `include_neighborhood_crowd` | [True, False] |

### `split_date = 2025-06-01` — justificación

- Junio es el único mes del período sin festivos nacionales → test limpio, sin días atípicos.
- 5 meses de train (83% del período) proporcionan historial individual denso para todos los usuarios.
- Proporciones train/test similares al 50mil (3 semanas / 1 semana ≈ 75%/25% en viajes).

### Por qué `holiday_dates` no era necesario en 50mil pero sí en 75mil

En marzo 2025 no hay ningún festivo nacional: todos los días son correctamente WD o WE sin necesidad
de pasar `holiday_dates`. Con el período enero–junio 2025 caen 7 festivos que, sin declarar, serían
tratados como laborables (WD), distorsionando las features TD/TSD/Crowd/LDA para esos días.
El parámetro `holiday_dates` ya existía en el pipeline — solo hay que rellenarlo.

---
## Hiperparámetros fijos — justificación cruzada v2 · v3 (50mil)

Los parámetros fijos se eligieron con el criterio de aparecer de forma **consistente en el top-15** de dos estudios independientes sobre el dataset 50mil:

- **v2** (`mdlf_marzo2025_v2.db`): 60 trials, LINE fijo desde `optuna_line_cercanias`.
- **v3** (`mdlf_marzo2025_v3_line.db`): 220 trials, LINE variable. Los parámetros confirmados en v2 pasaron a fijos en v3 — si v3 obtuvo mejor resultado con ellos fijos, la evidencia se considera validada.

El dataset de 75mil es distinto (6 meses vs. 1 mes, más usuarios), por lo que solo se fijan parámetros con señal muy clara en ambos estudios. Los que mostraron contradicción entre v2 y v3 se dejan variables.

| Parámetro | v2 top-15 | v3 top-15 | Valor fijo | Motivo |
|---|---|---|---|---|
| `norm` | **none 100%** | **none 100%** | `None` | Importancia fANOVA **92%** en v2; aplastante |
| `lda_use_periods` | **False 100%** | fijo en v3 ✓ | `False` | Periodos no aportan en 50mil; validado en v3 |
| `lda_token_granularity` | **temporal 100%** | fijo en v3 ✓ | `'temporal'` | Granularidad estación nunca gana; validado en v3 |
| `lda_alpha_none` | **False 100%** | fijo en v3 ✓ | `False` | Alpha siempre explícita; validado en v3 |
| `include_prox` | **True 100%** | fijo en v3 ✓ | `True` | Proximidad siempre suma; validado en v3 |
| `lr_scheduler` | **True 100%** | fijo en v3 ✓ | `True` | Scheduler siempre mejor; validado en v3 |
| `kernel_size` | **3: 93%** | fijo en v3 ✓ | `3` | 1 y 5 nunca ganan; validado en v3 |
| `cnn_channels` | **32: 93%** | **32: 100%** | `32` | Consistente en ambos estudios |
| `d_t` | **8: 80%** | **8: 100%** | `8` | Consistente en ambos estudios |

### Parámetros que siguen siendo variables — por qué

| Parámetro | v2 | v3 | Motivo |
|---|---|---|---|
| `optimizer` | sgd 93% | **adamw 100%** | Contradicción total — depende del régimen de entrenamiento |
| `include_neighborhood_crowd` | False 93% | **True 100%** | Contradicción total |
| `n_weeks` | 1–2 | **4: 100%** | Contradicción; con 6 meses de datos puede ser distinto |
| `n_heads` | 4: 80% | **2: 100%** | Contradicción |
| `n_topics` | 32/8 mezcla | 32/8 mezcla | No hay consenso claro |
| `lda_alpha`, `lda_beta` | continuos | continuos | Siempre variables |
| `lda_beta_none` | False 93% | False 87% | Evidencia moderada, no suficiente |
| `lda_log_counts` | True 53% | True 100% | v2 no es concluyente |
| `lda_min_df`, `lda_min_station_trips` | sin consenso v2 | clara en v3 | Un solo estudio no es suficiente |
| `lr`, `weight_decay` | continuos | continuos | Siempre variables |

In [ ]:
import os, sys, pickle
from pathlib import Path

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir), str(_mdlf_dir / 'LINE')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"CWD → {os.getcwd()}")

import optuna
import pandas as pd
import json
from datetime import date
from pipeline_mdlf import PipelineMDLF
from station_line import StationLINE

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ── Configuración fija ────────────────────────────────────────────────────────

ALMACEN      = '75mil_tarjetas_enero_junio_2025'
SPLIT_DATE   = '2025-06-01'   # junio completo como test (sin festivos)
N_EPOCHS_OPTUNA = 7
N_EPOCHS_FINAL  = 30
N_TRIALS        = 150

# v2: LINE fijo + 9 parámetros fijos por evidencia cruzada v2·v3 (50mil)
# (v1 = búsqueda completa con LINE variable — descartada por cambio de espacio)
STUDY_NAME = 'mdlf_junio2025_v2_line'
DB_PATH    = _mdlf_dir / 'runs' / f'{STUDY_NAME}.db'

# Festivos enero–junio 2025 que deben tratarse como WE.
HOLIDAY_DATES = frozenset([
    date(2025,  1,  1),   # Año Nuevo
    date(2025,  1,  6),   # Reyes Magos
    date(2025,  4, 17),   # Jueves Santo
    date(2025,  4, 18),   # Viernes Santo
    date(2025,  5,  1),   # Día del Trabajo
    date(2025,  5,  2),   # Comunidad de Madrid
    date(2025,  5, 15),   # San Isidro
])

# Grafo de cercanías — se carga una vez, se reutiliza en cada trial
_GRAPH_PATH = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo/check_transbordo/cercanias_multidigraph_sin_civis.pkl')
with open(_GRAPH_PATH, 'rb') as _f:
    G_cercanias = pickle.load(_f)
print(f"Grafo cercanías: {G_cercanias.number_of_nodes()} nodos | {G_cercanias.number_of_edges()} aristas")

_descomp = Path('/home/jupyter-lquijano/david/estudios/descomprimido')
TTP_PATHS = [str(_descomp / f'TTP_PERIODO_0{i}_2025.csv') for i in range(1, 7)]

OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

print(f"Almacén  : {ALMACEN}")
print(f"Split    : train < {SPLIT_DATE} | test >= {SPLIT_DATE}")
print(f"Festivos : {sorted(HOLIDAY_DATES)}")
print(f"Trials   : {N_TRIALS} × {N_EPOCHS_OPTUNA} épocas")
print(f"Study DB : {DB_PATH}")

In [ ]:
# ── Función objetivo ─────────────────────────────────────────────────────────

# ── LINE — fijos desde optuna_line_cercanias (score Spearman 0.7949) ─────────
LINE_DIM          = 64
LINE_ASYMMETRIC   = False
LINE_NUM_NEGATIVE = 2
LINE_LR           = 0.009611341424922943
LINE_FO_EPOCHS    = 850
LINE_SO_EPOCHS    = 200
LINE_INIT_SCALE   = 0.03825412692857836

# Entrenar LINE una sola vez (no depende de los hiperparámetros de Optuna)
_line_model = StationLINE(
    dim=LINE_DIM,
    num_negative=LINE_NUM_NEGATIVE,
    lr=LINE_LR,
    num_epochs_fo=LINE_FO_EPOCHS,
    num_epochs_so=LINE_SO_EPOCHS,
    init_scale=LINE_INIT_SCALE,
    asymmetric=LINE_ASYMMETRIC,
    random_state=42,
)
_line_model.fit_transform(G_cercanias)
_emb_ap_fixed = _line_model.to_punto_acceso(G_cercanias)
_LINE_FIXED_PATH = _mdlf_dir / 'runs' / 'line_fixed_optuna.parquet'
_emb_ap_fixed.to_parquet(_LINE_FIXED_PATH)
print(f"LINE fijo entrenado: {_emb_ap_fixed.shape[1]} dims → {_LINE_FIXED_PATH.name}")

_LINE_META = {
    'dim': LINE_DIM, 'asymmetric': LINE_ASYMMETRIC,
    'num_negative': LINE_NUM_NEGATIVE, 'lr': LINE_LR,
    'num_epochs_fo': LINE_FO_EPOCHS, 'num_epochs_so': LINE_SO_EPOCHS,
    'init_scale': LINE_INIT_SCALE,
}


def objective(trial: optuna.Trial) -> float:

    try:
        # ── LDA — variables ───────────────────────────────────────────────────
        # Fijos (100% top-15 v2+v3): lda_use_periods=False, lda_token_granularity='temporal',
        #                             lda_alpha_none=False (alpha siempre explícita)
        n_topics = trial.suggest_categorical('n_topics', [8, 16, 32, 48])
        lda_alpha = trial.suggest_float('lda_alpha', 0.001, 1.0, log=True)

        lda_beta_none = trial.suggest_categorical('lda_beta_none', [True, False])
        lda_beta = None if lda_beta_none else trial.suggest_float('lda_beta', 0.001, 1.0, log=True)

        lda_log_counts        = trial.suggest_categorical('lda_log_counts',        [True, False])
        lda_min_df            = trial.suggest_int('lda_min_df', 1, 5)
        lda_min_station_trips = trial.suggest_categorical('lda_min_station_trips', [0, 50, 100])

        # ── CNN — variables ───────────────────────────────────────────────────
        # Fijos (93-100% top-15 v2+v3): cnn_channels=32, d_t=8, kernel_size=3
        n_heads = trial.suggest_categorical('n_heads', [2, 4, 8])

        # ── Entrenamiento — variables ─────────────────────────────────────────
        # Fijos (100% top-15 v2+v3): lr_scheduler=True
        optimizer    = trial.suggest_categorical('optimizer',    ['sgd', 'adamw'])
        lr           = trial.suggest_float('lr', 1e-3, 1e-1, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
        n_weeks      = trial.suggest_int('n_weeks', 1, 8)

        # ── Extensiones — variables ───────────────────────────────────────────
        # Fijos (100% top-15 v2+v3): include_prox=True
        include_neighborhood_crowd = trial.suggest_categorical('include_neighborhood_crowd', [True, False])

        # ── Lanzar pipeline ───────────────────────────────────────────────────
        pipe = PipelineMDLF(
            nombre_almacen=ALMACEN,
            ttp_paths=TTP_PATHS,
            od_matrices_subdirs=OD_MATRICES_SUBDIRS,
            line_ap_path=_LINE_FIXED_PATH,
            split_date=SPLIT_DATE,
            holiday_dates=HOLIDAY_DATES,
            n_topics_cercanias=n_topics,
            lda_alpha=lda_alpha,
            lda_beta=lda_beta,
            lda_use_periods=False,             # fijo: 100% top-15 v2+v3
            lda_min_df=lda_min_df,
            lda_log_counts=lda_log_counts,
            lda_token_granularity='temporal',  # fijo: 100% top-15 v2+v3
            lda_min_station_trips=lda_min_station_trips,
            n_heads=n_heads,
            d_t=8,                             # fijo: 80% v2, 100% v3
            cnn_channels=32,                   # fijo: 93% v2, 100% v3
            kernel_size=3,                     # fijo: 93% v2, fijo v3
            optimizer=optimizer,
            lr=lr,
            weight_decay=weight_decay,
            lr_scheduler=True,                 # fijo: 100% top-15 v2+v3
            n_weeks=n_weeks,
            include_prox=True,                 # fijo: 100% top-15 v2+v3
            td_mode='context',
            include_neighborhood_crowd=include_neighborhood_crowd,
            norm=None,                         # fijo: 100% top-15 v2+v3, imp. 92%
            n_epochs=N_EPOCHS_OPTUNA,
            random_state=42,
            line_meta=_LINE_META,
        )

        pipe.run(run_name=f'optuna_v2_75mil_t{trial.number:03d}')
        trial.set_user_attr('top5', pipe.test_accuracy5)
        return pipe.test_accuracy

    finally:
        pass  # LINE ya no genera ficheros temporales por trial

In [ ]:
# ── Ejecutar estudio Optuna ───────────────────────────────────────────────────

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=40),
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=True,
)

trials_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Study    : {STUDY_NAME}")
print(f"DB       : {DB_PATH}")
print(f"Trials   : {trials_done} completados / {N_TRIALS} objetivo")
print()

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(f"\n✓ Estudio completado")
print(f"  Mejor Top-1 accuracy : {study.best_value:.4f}")
print(f"  Trial número         : {study.best_trial.number}")

In [ ]:
# ── Resultados ────────────────────────────────────────────────────────────────

print("=" * 60)
print("MEJORES HIPERPARÁMETROS")
print("=" * 60)
for k, v in study.best_trial.params.items():
    print(f"  {k:<40} {v}")
print(f"  {'Top-1 accuracy (7 épocas)':<40} {study.best_value:.4f}")

df_trials = study.trials_dataframe()
df_trials = df_trials.sort_values('value', ascending=False)
print(f"\nTop-10 trials:")
display(df_trials[['number', 'value', 'duration']].head(10))

In [ ]:
# ── Importancia de hiperparámetros ────────────────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    print("Importancia de hiperparámetros (fANOVA):")
    for param, imp in importances.items():
        bar = '█' * int(imp * 50)
        print(f"  {param:<40} {imp:.4f}  {bar}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

In [ ]:
# ── Reentrenamiento con mejor configuración y épocas completas ────────────────

best = study.best_trial.params

_lda_beta = None if best.get('lda_beta_none', True) else best.get('lda_beta')

# LINE: parámetros fijos (de optuna_line_cercanias)
print(f"LINE fijo (optuna_line_cercanias):")
print(f"  dim={LINE_DIM} | asymmetric={LINE_ASYMMETRIC} | num_negative={LINE_NUM_NEGATIVE}")
print(f"  lr={LINE_LR:.6f} | fo_epochs={LINE_FO_EPOCHS} | so_epochs={LINE_SO_EPOCHS} | init_scale={LINE_INIT_SCALE:.6f}")
print(f"  (embedding ya guardado en {_LINE_FIXED_PATH.name})")

print(f"\nReentrenando MDLF con {N_EPOCHS_FINAL} épocas...")

pipe_best = PipelineMDLF(
    nombre_almacen=ALMACEN,
    ttp_paths=TTP_PATHS,
    od_matrices_subdirs=OD_MATRICES_SUBDIRS,
    line_ap_path=_LINE_FIXED_PATH,
    split_date=SPLIT_DATE,
    holiday_dates=HOLIDAY_DATES,
    n_topics_cercanias=best['n_topics'],
    lda_alpha=best['lda_alpha'],
    lda_beta=_lda_beta,
    lda_use_periods=False,
    lda_min_df=best['lda_min_df'],
    lda_log_counts=best['lda_log_counts'],
    lda_token_granularity='temporal',
    lda_min_station_trips=best['lda_min_station_trips'],
    n_heads=best['n_heads'],
    d_t=8,
    cnn_channels=32,
    kernel_size=3,
    optimizer=best['optimizer'],
    lr=best['lr'],
    weight_decay=best['weight_decay'],
    lr_scheduler=True,
    n_weeks=best['n_weeks'],
    include_prox=True,
    td_mode='context',
    include_neighborhood_crowd=best['include_neighborhood_crowd'],
    norm=None,
    n_epochs=N_EPOCHS_FINAL,
    random_state=42,
    line_meta=_LINE_META,
)

pipe_best.run(run_name='optuna_v2_75mil_best_final')

print(f"\n{'='*50}")
print(f"RESULTADO FINAL (best config, {N_EPOCHS_FINAL} épocas)")
print(f"  Top-1 accuracy : {pipe_best.test_accuracy:.4f}")
print(f"  Top-5 accuracy : {pipe_best.test_accuracy5:.4f}")
print(f"  (Proxy 7 épocas era: {study.best_value:.4f})")

In [ ]:
# ── Guardar mejores params en JSON ────────────────────────────────────────────

_out = {
    'study_name': STUDY_NAME,
    'almacen': ALMACEN,
    'split_date': SPLIT_DATE,
    'holiday_dates': [str(d) for d in sorted(HOLIDAY_DATES)],
    'best_trial': study.best_trial.number,
    'top1_proxy_7epochs': study.best_value,
    'top1_final_30epochs': pipe_best.test_accuracy,
    'top5_final_30epochs': pipe_best.test_accuracy5,
    'fixed_params': {
        'line_dim': LINE_DIM, 'line_asymmetric': LINE_ASYMMETRIC,
        'line_num_negative': LINE_NUM_NEGATIVE, 'line_lr': LINE_LR,
        'line_num_epochs_fo': LINE_FO_EPOCHS, 'line_num_epochs_so': LINE_SO_EPOCHS,
        'line_init_scale': LINE_INIT_SCALE,
        'norm': None, 'lda_use_periods': False, 'lda_token_granularity': 'temporal',
        'lda_alpha_none': False, 'include_prox': True, 'lr_scheduler': True,
        'kernel_size': 3, 'cnn_channels': 32, 'd_t': 8,
    },
    'best_params': study.best_trial.params,
}

_json_path = _mdlf_dir / 'runs' / f'{STUDY_NAME}_best.json'
with open(_json_path, 'w') as f:
    json.dump(_out, f, indent=2, default=str)

print(f"Resultados guardados en: {_json_path}")

In [ ]:
# ── Top combinaciones de hiperparámetros ──────────────────────────────────────

import pandas as pd

df_trials = study.trials_dataframe()
df_ok = df_trials[df_trials['state'] == 'COMPLETE'].copy()

param_cols = [c for c in df_ok.columns if c.startswith('params_')]
param_display = {c: c.replace('params_', '') for c in param_cols}

top_n = 15
df_top = (
    df_ok
    .sort_values('value', ascending=False)
    .head(top_n)
    [['number', 'value', 'duration'] + param_cols]
    .rename(columns={'number': 'trial', 'value': 'top1_proxy', **param_display})
    .reset_index(drop=True)
)

print(f'Top {top_n} trials (de {len(df_ok)} completados) — proxy 7 épocas')
print(f'Mejor Top-1 proxy : {df_ok["value"].max():.4f}')
print(f'Mediana           : {df_ok["value"].median():.4f}')
print()
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 10 else f'{x:.2f}')
display(df_top)

key_params = [
    'line_dim', 'line_asymmetric', 'line_num_negative',
    'n_topics', 'lda_alpha', 'lda_log_counts', 'lda_min_df',
    'n_heads', 'd_t', 'cnn_channels',
    'optimizer', 'lr', 'n_weeks', 'include_neighborhood_crowd',
]
available = ['trial', 'top1_proxy'] + [p for p in key_params if p in df_top.columns]

print()
print('── Parámetros clave ──────────────────────────────────────────────────────')
display(df_top[available])

## Top 15 combinaciones — v1 75mil (carga directa desde DB)

In [ ]:
# ── Cargar estudio desde DB (standalone) ──────────────────────────────────────
import optuna, pandas as pd, warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF')
_db = _mdlf_dir / 'runs' / 'mdlf_junio2025_v1_line.db'

_study = optuna.load_study(
    study_name='mdlf_junio2025_v1_line',
    storage=f'sqlite:///{_db}',
)

_df = _study.trials_dataframe()
_df_ok = _df[_df['state'] == 'COMPLETE'].copy()

_param_cols = [c for c in _df_ok.columns if c.startswith('params_')]
_df_ok = _df_ok.rename(columns={c: c.replace('params_', '') for c in _param_cols})
if 'user_attrs_top5' in _df_ok.columns:
    _df_ok = _df_ok.rename(columns={'user_attrs_top5': 'top5'})
_df_ok = _df_ok.rename(columns={'number': 'trial', 'value': 'top1'})

TOP_N = 15
_df_top15 = (
    _df_ok
    .sort_values('top1', ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
_df_top15.index += 1

_has_top5 = 'top5' in _df_top15.columns

print(f"Study  : mdlf_junio2025_v1_line")
print(f"Trials : {len(_df_ok)} completados de {len(_df)} total  (7 épocas proxy)")
print(f"Mejor  : top1={_df_ok['top1'].max():.4f}" + (f"  top5={_df_ok['top5'].max():.4f}" if _has_top5 else "  (top5 no disponible)") + f"  (trial #{int(_df_top15.loc[1,'trial'])})")
print(f"Mediana top1: {_df_ok['top1'].median():.4f}")

In [ ]:
# ── Top 15 — todos los hiperparámetros, agrupados por sección ─────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 20)

_score_cols = ['trial', 'top1'] + (['top5'] if _has_top5 else [])
_base = _df_top15[_score_cols].copy()
_fmt_base = {'top1': '{:.4f}', **({'top5': '{:.4f}'} if _has_top5 else {})}

# ── LINE ──────────────────────────────────────────────────────────────────────
_line_cols = ['line_dim', 'line_asymmetric', 'line_num_negative',
              'line_lr', 'line_fo_epochs', 'line_so_epochs', 'line_init_scale']
_line_cols = [c for c in _line_cols if c in _df_top15.columns]
print("━" * 80)
print(f"  TOP {TOP_N}  —  LINE")
print("━" * 80)
display(_base.join(_df_top15[_line_cols]).style.format({
    **_fmt_base,
    'line_lr': '{:.5f}',
}))

# ── LDA ───────────────────────────────────────────────────────────────────────
_lda_cols = ['n_topics', 'lda_alpha_none', 'lda_alpha', 'lda_beta_none', 'lda_beta',
             'lda_log_counts', 'lda_min_df', 'lda_min_station_trips',
             'lda_use_periods', 'lda_token_granularity']
_lda_cols = [c for c in _lda_cols if c in _df_top15.columns]
print()
print("━" * 80)
print(f"  TOP {TOP_N}  —  LDA")
print("━" * 80)
display(_base.join(_df_top15[_lda_cols]).style.format({
    **_fmt_base,
    'lda_alpha': lambda x: f'{x:.5f}' if pd.notna(x) else '—',
    'lda_beta':  lambda x: f'{x:.5f}' if pd.notna(x) else '—',
}))

# ── CNN + Train + Extras ───────────────────────────────────────────────────────
_rest_cols = ['n_heads', 'd_t', 'cnn_channels', 'kernel_size',
              'optimizer', 'lr', 'weight_decay', 'lr_scheduler', 'n_weeks',
              'include_prox', 'include_neighborhood_crowd', 'norm']
_rest_cols = [c for c in _rest_cols if c in _df_top15.columns]
print()
print("━" * 80)
print(f"  TOP {TOP_N}  —  CNN + Train + Extras")
print("━" * 80)
display(_base.join(_df_top15[_rest_cols]).style.format({
    **_fmt_base,
    'lr':           '{:.5f}',
    'weight_decay': '{:.5f}',
}))